# 📊 Datathon Template — TABULAR DATA (Classification + Regression)

**পুরো workflow (Kaggle-style):**

| # | Step | কি করব |
|---|------|--------|
| 0 | Setup & Config | library import, path, TARGET, metric সেট |
| 1 | Load Data | train / test / sample_submission পড়া |
| 2 | ⚡ Quick Baseline | ৫ মিনিটে প্রথম submission |
| 3 | First Look | shape, dtype, missing, unique |
| 4 | Data Cleaning | placeholder → NaN, type fix, duplicate, constant column, high-missing column |
| 5 | EDA | target, univariate, bivariate, correlation, train-vs-test |
| 6 | Outliers | IQR, z-score, capping, IsolationForest |
| 7 | Feature Engineering | date features, log transform, ratio, frequency, group-agg |
| 8 | Encoding & Scaling | one-hot, ordinal, target encoding, scaler comparison |
| 9 | Feature Selection | variance, correlation, mutual info, RFE, L1, importance |
| 10 | Dimensionality Reduction | PCA, t-SNE |
| 11 | Model Zoo | ১৫+ model একসাথে CV দিয়ে compare |
| 12 | Imbalance | class_weight, SMOTE |
| 13 | Hyperparameter Tuning | RandomizedSearchCV, Optuna |
| 14 | K-Fold Training + OOF | final model, test prediction |
| 15 | Ensembling | average, weighted, stacking, threshold tuning |
| 16 | Evaluation Plots | confusion matrix, ROC, PR, residual, learning curve |
| 17 | Interpretation | feature importance, permutation, SHAP, PDP |
| 18 | Submission | CSV বানানো + sanity check |

> **ব্যবহারের নিয়ম:** শুধু **Section 0 এর CONFIG** অংশ নিজের dataset অনুযায়ী বদলাও, তারপর উপর থেকে নিচে Run করো।
> কোনো data না দিলে notebook নিজেই একটা **messy demo dataset** (missing, duplicate, outlier সহ) বানিয়ে নেবে — practice এর জন্য।

## ☁️ Google Colab Setup (সবার আগে এটা চালাও)

**GPU চালু করা:** `Runtime → Change runtime type → T4 GPU → Save` (image / transformer এর জন্য জরুরি)

**Data আনার ৪টা উপায় — নিচের `DATA_SOURCE` বদলাও:**
| `DATA_SOURCE` | কিভাবে |
|---|---|
| `"demo"` | কিছু লাগবে না — notebook নিজেই practice data বানাবে |
| `"upload"` | Cell চালালে **Choose Files** button আসবে → `train.csv`, `test.csv`, `sample_submission.csv` (বা একটা `.zip`) select করো |
| `"drive"` | আগে Google Drive এ data রাখো (যেমন `MyDrive/datathon/tabular/`) → `DRIVE_DATA_DIR` ঠিক করো → permission দাও |
| `"kaggle"` | Kaggle → Settings → **Create New Token** → `kaggle.json` download → cell চালালে upload করো; competition page এ আগে **rules accept** করতে হবে |

সব data শেষে `data/tabular` folder এ আসবে, `.zip` থাকলে auto unzip হবে — তাই নিচের CONFIG এ path বদলাতে হবে না
(শুধু file / column এর নাম মিলিয়ে নিও)।

> ⚠️ Colab বন্ধ হলে `/content` এর সব file মুছে যায় — submission আর model notebook এর শেষ cell দিয়ে download / Drive এ save করো।

In [ ]:
import sys, os, shutil, zipfile, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# ---------- 1) যে library Colab এ নেই সেগুলো install ----------
NEED = {"seaborn": "seaborn", "lightgbm": "lightgbm", "xgboost": "xgboost", "catboost": "catboost", "optuna": "optuna", "shap": "shap", "imblearn": "imbalanced-learn", "missingno": "missingno"}          # {import_name: pip_name}
missing = [pip for mod, pip in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

# ---------- 2) Data ----------
DATA_SOURCE = "demo"                                  # "demo" | "upload" | "drive" | "kaggle"
DRIVE_DATA_DIR = "/content/drive/MyDrive/datathon/tabular"
KAGGLE_COMPETITION = ""                               # e.g. "titanic"  (URL: kaggle.com/competitions/<এই নাম>)
KAGGLE_DATASET = ""                                   # competition না হয়ে dataset হলে: "owner/dataset-name"
LOCAL_DATA_DIR = Path("data/tabular")

def unzip_all(folder):
    while zips := list(Path(folder).rglob("*.zip")):                       # zip এর ভেতরে zip থাকলেও (Kaggle এ common)
        for z in zips:
            print("Unzipping", z.name)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(z.parent)
            z.unlink()

if IN_COLAB and DATA_SOURCE != "demo":
    from google.colab import files, drive
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if DATA_SOURCE == "upload":
        for name in files.upload():
            shutil.move(name, LOCAL_DATA_DIR / name)
    elif DATA_SOURCE == "drive":
        drive.mount("/content/drive")
        shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR, dirs_exist_ok=True)   # local copy → Drive থেকে পড়ার চেয়ে অনেক দ্রুত
    elif DATA_SOURCE == "kaggle":
        # নতুন token (KGAT_...) বা পুরনো kaggle.json — যেটা আছে সেটা দিয়েই চলবে
        import getpass
        if not os.environ.get("KAGGLE_API_TOKEN") and not Path("~/.kaggle/kaggle.json").expanduser().exists():
            tok = getpass.getpass("Kaggle API token paste করো (লুকানো থাকবে, KGAT_... দিয়ে শুরু) অথবা খালি রেখে Enter দিয়ে kaggle.json upload করো: ").strip()
            if tok:
                os.environ["KAGGLE_API_TOKEN"] = tok
            else:
                files.upload()
                Path("~/.kaggle").expanduser().mkdir(exist_ok=True)
                shutil.move("kaggle.json", Path("~/.kaggle/kaggle.json").expanduser())
                os.chmod(Path("~/.kaggle/kaggle.json").expanduser(), 0o600)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=False)
        if KAGGLE_COMPETITION:
            subprocess.run(["kaggle", "competitions", "download", "-c", KAGGLE_COMPETITION, "-p", str(LOCAL_DATA_DIR)], check=True)
        else:
            subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(LOCAL_DATA_DIR)], check=True)
    unzip_all(LOCAL_DATA_DIR)

if LOCAL_DATA_DIR.exists():
    print(f"\n📁 {LOCAL_DATA_DIR} এর ভেতরে:")
    for p in sorted(LOCAL_DATA_DIR.iterdir())[:30]:
        print("  ", p.name, "/" if p.is_dir() else f"({p.stat().st_size / 1e6:.2f} MB)")
print("IN_COLAB =", IN_COLAB, "| DATA_SOURCE =", DATA_SOURCE)

## 0. Setup & Config

In [ ]:
import os, re, time, random, warnings, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold, GroupKFold,
                                     TimeSeriesSplit, cross_validate, cross_val_predict,
                                     RandomizedSearchCV, learning_curve)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer,
                                   QuantileTransformer, OneHotEncoder, OrdinalEncoder, LabelEncoder,
                                   TargetEncoder)
from sklearn.feature_selection import (VarianceThreshold, mutual_info_classif, mutual_info_regression,
                                       SelectKBest, f_classif, f_regression, RFECV, SelectFromModel)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.ensemble import IsolationForest
from sklearn import metrics
from sklearn.base import clone

# --- Models ---
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import (LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier,
                              ExtraTreesRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              AdaBoostClassifier, AdaBoostRegressor)
from sklearn.neural_network import MLPClassifier, MLPRegressor


def try_import(name):
    """Optional library — না থাকলে None দেবে, notebook crash করবে না।"""
    try:
        return importlib.import_module(name)
    except ImportError:
        print(f"⚠️  '{name}' installed নেই — এর অংশ skip হবে (pip install {name})")
        return None

xgb = try_import("xgboost")
lgb = try_import("lightgbm")
cb = try_import("catboost")
optuna = try_import("optuna")
shap = try_import("shap")
imblearn = try_import("imblearn")
msno = try_import("missingno")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100

In [ ]:
# ======================= CONFIG — শুধু এখানে বদলাও =======================
DATA_DIR = Path("data/tabular")              # Kaggle: Path("/kaggle/input/<competition-name>")
TRAIN_FILE = DATA_DIR / "train.csv"
TEST_FILE = DATA_DIR / "test.csv"
SUBMISSION_FILE = DATA_DIR / "sample_submission.csv"
OUTPUT_DIR = Path("outputs/tabular")

TARGET = "target"         # target column এর নাম
ID_COL = "id"             # ID column (না থাকলে None)
TASK = "auto"             # "classification" / "regression" / "auto"
METRIC = "auto"           # sklearn scoring name: "accuracy", "f1_macro", "roc_auc", "neg_root_mean_squared_error", "r2" ...
                          # competition যে metric এ judge করবে সেটাই দাও!

SEED = 42
N_FOLDS = 5
MISSING_DROP_THRESH = 0.60     # এর বেশি missing থাকলে column drop
HIGH_CARD_THRESH = 15          # এর বেশি unique category হলে one-hot না করে target/ordinal encoding
SLOW_MODEL_MAX_ROWS = 20_000   # এর বেশি row হলে SVM/KNN skip (খুব slow)
CAP_OUTLIERS = True            # 1%-99% percentile এ clip করবে
N_OPTUNA_TRIALS = 30

DEMO_TASK = "classification"   # data না থাকলে demo dataset: "classification" / "regression"
# ========================================================================

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### (Practice) Messy demo dataset generator
নিজের data থাকলে এই cell কিছুই করবে না। Data না থাকলে একটা fake "customer churn" dataset বানাবে
যেখানে ইচ্ছা করে **missing value, '?' placeholder, duplicate row, outlier, constant column, date string** রাখা আছে।

In [ ]:
def make_demo_tabular(task="classification", n=4000, seed=SEED):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "id": np.arange(n),
        "age": rng.integers(18, 75, n).astype(float),
        "income": np.round(rng.lognormal(10.5, 0.6, n), 2),
        "city": rng.choice(["Dhaka", "Chattogram", "Khulna", "Rajshahi", "Sylhet", "Barishal"], n,
                           p=[.35, .2, .15, .12, .1, .08]),
        "gender": rng.choice(["Male", "Female"], n),
        "plan": rng.choice(["basic", "standard", "premium"], n, p=[.5, .3, .2]),
        "num_products": rng.poisson(2, n),
        "tenure_months": rng.integers(0, 120, n),
        "monthly_spend": np.round(rng.gamma(2, 30, n), 2),
        "support_calls": rng.poisson(1.5, n),
        "signup_date": (pd.Timestamp("2018-01-01") + pd.to_timedelta(rng.integers(0, 2000, n), unit="D")).astype(str),
        "satisfaction": rng.integers(1, 6, n),
        "country": "Bangladesh",                       # constant column (useless)
        "notes": np.where(rng.random(n) < 0.1, "vip", None),  # 90% missing
    })
    plan_num = df["plan"].map({"basic": 0, "standard": 1, "premium": 2})
    if task == "classification":
        logit = (-0.3 + 0.55 * df.support_calls - 0.025 * df.tenure_months - 0.5 * plan_num
                 + 0.008 * df.monthly_spend - 0.35 * (df.satisfaction - 3) + 0.4 * (df.city == "Dhaka")
                 + rng.normal(0, 0.5, n))
        df["target"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    else:
        df["target"] = np.round(50 * df.tenure_months + 3 * df.monthly_spend * (1 + plan_num)
                                + 0.002 * df.income + 200 * df.satisfaction + rng.normal(0, 300, n), 2)

    # ---- make it messy ----
    for col, frac in [("age", .08), ("income", .10), ("city", .05), ("monthly_spend", .03)]:
        df.loc[rng.random(n) < frac, col] = np.nan
    df["satisfaction"] = df["satisfaction"].astype(str)
    df.loc[rng.random(n) < .05, "satisfaction"] = "?"                       # placeholder
    out_idx = rng.choice(n, n // 100, replace=False)
    df.loc[out_idx, "income"] = df.loc[out_idx, "income"] * 25               # outliers
    df.loc[rng.choice(n, 10, replace=False), "age"] = 150                    # impossible values

    train_df, test_df = train_test_split(df, test_size=0.25, random_state=seed)
    dups = train_df.sample(60, random_state=seed).assign(id=lambda d: d["id"])  # exact duplicate rows
    train_df = pd.concat([train_df, dups]).sample(frac=1, random_state=seed)
    sub = pd.DataFrame({"id": test_df["id"], "target": 0})
    return train_df, test_df.drop(columns="target"), sub


if not TRAIN_FILE.exists():
    print(f"'{TRAIN_FILE}' পাওয়া যায়নি → DEMO dataset বানানো হচ্ছে ({DEMO_TASK})")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _tr, _te, _sub = make_demo_tabular(DEMO_TASK)
    _tr.to_csv(TRAIN_FILE, index=False); _te.to_csv(TEST_FILE, index=False); _sub.to_csv(SUBMISSION_FILE, index=False)

## 1. Load Data
অন্য format হলে: `pd.read_excel()`, `pd.read_parquet()`, `pd.read_json()`, `pd.read_csv(sep=";")`

In [ ]:
train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE) if TEST_FILE.exists() else None
sample_sub = pd.read_csv(SUBMISSION_FILE) if SUBMISSION_FILE.exists() else None

print("train:", train.shape, "| test:", None if test is None else test.shape,
      "| sample_sub:", None if sample_sub is None else sample_sub.shape)
display(train.head())
if sample_sub is not None:
    display(sample_sub.head(3))

In [ ]:
# ---- Task & metric auto-detect ----
if TASK == "auto":
    TASK = "classification" if (not pd.api.types.is_numeric_dtype(train[TARGET]) or train[TARGET].nunique() <= 20) else "regression"
N_CLASSES = train[TARGET].nunique() if TASK == "classification" else None
IS_BINARY = N_CLASSES == 2

if METRIC == "auto":
    METRIC = ("roc_auc" if IS_BINARY else "f1_macro") if TASK == "classification" else "neg_root_mean_squared_error"
print(f"TASK = {TASK} | N_CLASSES = {N_CLASSES} | METRIC = {METRIC}")

## 2. ⚡ Quick Baseline (সময় কম থাকলে আগে এটা submit করো)
HistGradientBoosting নিজেই NaN handle করে, categorical গুলো শুধু category code বানিয়ে দিলেই চলে।
এটা একটা **safety submission** — পরে improve করবে।

In [ ]:
def quick_baseline(train, test, target=TARGET, id_col=ID_COL):
    X = train.drop(columns=[c for c in [target, id_col] if c]).copy()
    Xt = test.drop(columns=[c for c in [id_col] if c]).copy() if test is not None else None
    for c in X.select_dtypes(exclude="number").columns:
        cats = pd.Categorical(pd.concat([X[c], Xt[c]]) if Xt is not None else X[c]).categories
        X[c] = pd.Categorical(X[c], categories=cats).codes
        if Xt is not None:
            Xt[c] = pd.Categorical(Xt[c], categories=cats).codes
    model = (HistGradientBoostingClassifier if TASK == "classification" else HistGradientBoostingRegressor)(random_state=SEED)
    score = cross_validate(model, X, train[target], cv=5, scoring=METRIC)["test_score"]
    print(f"Quick baseline CV {METRIC}: {score.mean():.4f} ± {score.std():.4f}")
    if Xt is not None:
        model.fit(X, train[target])
        pred = model.predict_proba(Xt)[:, 1] if METRIC == "roc_auc" else model.predict(Xt)
        pd.DataFrame({id_col: test[id_col], target: pred}).to_csv(OUTPUT_DIR / "submission_quick_baseline.csv", index=False)
        print("Saved →", OUTPUT_DIR / "submission_quick_baseline.csv")

quick_baseline(train, test)

## 3. First Look — data কেমন?

In [ ]:
def overview(df):
    """প্রতিটা column এর dtype, missing, unique, sample value এক টেবিলে।"""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
        "sample": [df[c].dropna().unique()[:4].tolist() for c in df.columns],
    }).sort_values("pct_missing", ascending=False)

print(f"Rows: {len(train):,} | Cols: {train.shape[1]} | Memory: {train.memory_usage(deep=True).sum()/1e6:.2f} MB")
display(overview(train))

In [ ]:
train.info()
display(train.describe().T)                       # numeric summary
if len(train.select_dtypes(exclude="number").columns):
    display(train.describe(exclude="number").T)   # categorical summary

## 4. Data Cleaning
### 4.1 Column name clean + placeholder → NaN + type fix

In [ ]:
def clean_name(c):
    return re.sub(r"[^0-9a-zA-Z_]+", "_", str(c).strip()).strip("_")

train.columns = [clean_name(c) for c in train.columns]
if test is not None:
    test.columns = [clean_name(c) for c in test.columns]
TARGET = clean_name(TARGET)
ID_COL = clean_name(ID_COL) if ID_COL else None

PLACEHOLDERS = ["?", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "none", "", " ", "-", "--", "missing", "unknown", "Unknown"]

def is_text_col(s):
    return pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s)

def fix_types(df, min_numeric_ratio=0.9):
    df = df.copy()
    obj_cols = [c for c in df.columns if is_text_col(df[c])]
    for c in obj_cols:
        df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v).replace(PLACEHOLDERS, np.nan)
    for c in obj_cols:
        converted = pd.to_numeric(df[c], errors="coerce")
        if df[c].notna().sum() and converted.notna().sum() / df[c].notna().sum() >= min_numeric_ratio:
            df[c] = converted                    # "12", "3.5" → number
            print(f"  '{c}' → numeric")
    return df

train = fix_types(train)
if test is not None:
    test = fix_types(test)

# Domain knowledge দিয়ে impossible value → NaN (নিজের data অনুযায়ী বদলাও)
VALID_RANGES = {"age": (0, 110)}
for col, (lo, hi) in VALID_RANGES.items():
    for df in [train, test]:
        if df is not None and col in df:
            bad = ~df[col].between(lo, hi) & df[col].notna()
            print(f"  {col}: {bad.sum()} impossible values → NaN")
            df.loc[bad, col] = np.nan

# Target এ NaN থাকলে সেই row বাদ
train = train.dropna(subset=[TARGET]).reset_index(drop=True)

### 4.2 Duplicate rows

In [ ]:
feature_cols_for_dup = [c for c in train.columns if c not in [ID_COL]]
n_dup = train.duplicated(subset=feature_cols_for_dup).sum()
print(f"Exact duplicate rows (ID বাদে): {n_dup}")

# একই features কিন্তু ভিন্ন target = noisy/conflicting label
feat_only = [c for c in feature_cols_for_dup if c != TARGET]
conflict = train[train.duplicated(subset=feat_only, keep=False)].groupby(feat_only, dropna=False)[TARGET].nunique()
print(f"Conflicting duplicates (same X, different y): {(conflict > 1).sum()}")

train = train.drop_duplicates(subset=feature_cols_for_dup).reset_index(drop=True)
if ID_COL:
    train = train.drop_duplicates(subset=[ID_COL]).reset_index(drop=True)
print("After dedup:", train.shape)

### 4.3 Missing values — analysis

In [ ]:
miss = (train.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
display(miss.to_frame("pct_missing"))

if len(miss):
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    miss.plot.bar(ax=axes[0], color="salmon"); axes[0].set_title("Missing %  per column"); axes[0].axhline(MISSING_DROP_THRESH * 100, ls="--", c="k")
    sns.heatmap(train[miss.index].isna().iloc[:500], cbar=False, ax=axes[1], cmap="viridis"); axes[1].set_title("Missing pattern (first 500 rows)")
    plt.tight_layout(); plt.show()

if msno is not None and len(miss):
    msno.matrix(train[miss.index], figsize=(12, 3), fontsize=9); plt.show()

# Missing-ness কি target এর সাথে related? (বড় পার্থক্য হলে missing indicator feature কাজে লাগে)
if pd.api.types.is_numeric_dtype(train[TARGET]):
    print("Target mean — value missing vs present:")
    for c in [c for c in miss.index if c != TARGET][:8]:
        print(f"  {c:20s} missing: {train.loc[train[c].isna(), TARGET].mean():.3f} | present: {train.loc[train[c].notna(), TARGET].mean():.3f}")

### 4.4 Missing values — handling
**Strategy (rule of thumb):**
- > 60% missing → column **drop** (অথবা শুধু `is_missing` flag রাখো)
- Numeric → **median** (outlier robust) / mean / KNNImputer / IterativeImputer (MICE)
- Categorical → **mode** অথবা `"missing"` নামে আলাদা category
- Group-wise impute → যেমন city অনুযায়ী income এর median
- Tree model (LightGBM/XGBoost/CatBoost/HistGB) নিজেই NaN handle করতে পারে

> ⚠️ **Leakage এড়াতে:** imputer শুধু train fold এ `fit` করে validation/test এ `transform` — তাই নিচে আসল imputation
> **Pipeline এর ভেতরে** (Section 8) করা হয়েছে। এখানে শুধু toolbox দেখানো হলো।

In [ ]:
drop_missing_cols = [c for c in miss.index if miss[c] > MISSING_DROP_THRESH * 100 and c != TARGET]
print("Dropping high-missing columns:", drop_missing_cols)

# Missing indicator — drop করার আগে flag রেখে দাও (কখনো কখনো খুব useful)
for c in drop_missing_cols:
    for df in [train, test]:
        if df is not None:
            df[f"{c}_is_missing"] = df[c].isna().astype(int)
train = train.drop(columns=drop_missing_cols)
if test is not None:
    test = test.drop(columns=drop_missing_cols, errors="ignore")

# ---------- Toolbox (demo, original data বদলাচ্ছে না) ----------
_num = train.select_dtypes("number").drop(columns=[c for c in [TARGET, ID_COL] if c in train], errors="ignore")
_demo = {
    "median": SimpleImputer(strategy="median").fit_transform(_num),
    "knn(k=5)": KNNImputer(n_neighbors=5).fit_transform(_num.iloc[:2000]),
    "iterative(MICE)": IterativeImputer(random_state=SEED, max_iter=5).fit_transform(_num.iloc[:2000]),
}
for k, v in _demo.items():
    print(f"{k:16s} → shape {v.shape}, NaN left: {np.isnan(v).sum()}")

# Group-wise impute example: city অনুযায়ী income median
if {"income", "city"} <= set(train.columns):
    _gi = train.groupby("city")["income"].transform("median")
    print("Group-wise imputed income (example):", train["income"].fillna(_gi).isna().sum(), "NaN left")

### 4.5 Constant / quasi-constant / ID-like columns

In [ ]:
def useless_columns(df, quasi_thresh=0.99):
    out = []
    for c in df.columns:
        if c in [TARGET, ID_COL]:
            continue
        vc = df[c].value_counts(normalize=True, dropna=False)
        if df[c].nunique(dropna=False) <= 1:
            out.append((c, "constant"))
        elif vc.iloc[0] >= quasi_thresh:
            out.append((c, f"quasi-constant ({vc.iloc[0]:.1%})"))
        elif is_text_col(df[c]) and df[c].nunique() == len(df):
            out.append((c, "ID-like (all unique)"))
    return out

useless = useless_columns(train)
print(useless)
drop_cols = [c for c, _ in useless]
train = train.drop(columns=drop_cols)
if test is not None:
    test = test.drop(columns=drop_cols, errors="ignore")

## 5. Exploratory Data Analysis (EDA)

In [ ]:
def col_groups(df):
    feats = [c for c in df.columns if c not in [TARGET, ID_COL]]
    num = [c for c in feats if pd.api.types.is_numeric_dtype(df[c])]
    cat = [c for c in feats if c not in num]
    return feats, num, cat

FEATURES, NUM_COLS, CAT_COLS = col_groups(train)
print(f"Numeric ({len(NUM_COLS)}): {NUM_COLS}\nCategorical ({len(CAT_COLS)}): {CAT_COLS}")

def grid(cols, ncols=3, size=(5, 3.2)):
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size[0] * ncols, size[1] * nrows))
    axes = np.array(axes).reshape(-1)
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    return fig, axes

### 5.1 Target distribution
Classification → class imbalance আছে কিনা। Regression → skewed কিনা (skewed হলে `log1p` target try করো)।

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
if TASK == "classification":
    vc = train[TARGET].value_counts()
    sns.countplot(x=TARGET, data=train, ax=axes[0], order=vc.index)
    for p in axes[0].patches:
        axes[0].annotate(f"{p.get_height() / len(train):.1%}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
    axes[1].pie(vc, labels=vc.index, autopct="%1.1f%%"); axes[1].set_title("Class share")
    print("Imbalance ratio (max/min):", round(vc.max() / vc.min(), 2))
else:
    sns.histplot(train[TARGET], kde=True, ax=axes[0]); axes[0].set_title(f"{TARGET} (skew={train[TARGET].skew():.2f})")
    sns.histplot(np.log1p(train[TARGET].clip(lower=0)), kde=True, ax=axes[1], color="g"); axes[1].set_title("log1p(target)")
plt.tight_layout(); plt.show()

### 5.2 Univariate — numeric distributions + skewness

In [ ]:
if NUM_COLS:
    fig, axes = grid(NUM_COLS)
    for ax, c in zip(axes, NUM_COLS):
        sns.histplot(train[c], kde=True, ax=ax, bins=40)
        ax.set_title(f"{c} | skew={train[c].skew():.2f}")
    plt.tight_layout(); plt.show()

    skew = train[NUM_COLS].skew().sort_values(key=abs, ascending=False)
    display(skew.to_frame("skew").style.background_gradient(cmap="coolwarm"))

In [ ]:
# Boxplots → outlier চোখে দেখা
if NUM_COLS:
    fig, axes = grid(NUM_COLS, ncols=4, size=(4, 2.5))
    for ax, c in zip(axes, NUM_COLS):
        sns.boxplot(x=train[c], ax=ax, color="lightblue"); ax.set_title(c)
    plt.tight_layout(); plt.show()

### 5.3 Univariate — categorical

In [ ]:
if CAT_COLS:
    fig, axes = grid(CAT_COLS)
    for ax, c in zip(axes, CAT_COLS):
        order = train[c].value_counts().index[:15]
        sns.countplot(y=c, data=train, order=order, ax=ax)
        ax.set_title(f"{c} ({train[c].nunique()} unique)")
    plt.tight_layout(); plt.show()

### 5.4 Bivariate — feature vs target

In [ ]:
if NUM_COLS:
    fig, axes = grid(NUM_COLS)
    for ax, c in zip(axes, NUM_COLS):
        if TASK == "classification":
            sns.boxplot(x=TARGET, y=c, data=train, ax=ax)
        else:
            sns.regplot(x=c, y=TARGET, data=train.sample(min(2000, len(train)), random_state=SEED),
                        ax=ax, scatter_kws={"s": 6, "alpha": .4}, line_kws={"color": "red"})
        ax.set_title(f"{c} vs {TARGET}")
    plt.tight_layout(); plt.show()

In [ ]:
# Categorical → প্রতিটা category তে target এর গড় (classification এ = positive rate)
if CAT_COLS and pd.api.types.is_numeric_dtype(train[TARGET]):
    fig, axes = grid(CAT_COLS)
    for ax, c in zip(axes, CAT_COLS):
        order = train[c].value_counts().index[:15]
        sns.barplot(y=c, x=TARGET, data=train, order=order, ax=ax, errorbar=("ci", 95))
        ax.set_title(f"mean {TARGET} by {c}")
    plt.tight_layout(); plt.show()

In [ ]:
# Classification: KDE per class (কোন feature class গুলোকে আলাদা করে?)
if TASK == "classification" and NUM_COLS:
    fig, axes = grid(NUM_COLS)
    for ax, c in zip(axes, NUM_COLS):
        sns.kdeplot(data=train, x=c, hue=TARGET, common_norm=False, fill=True, alpha=.3, ax=ax)
    plt.tight_layout(); plt.show()

### 5.5 Correlation

In [ ]:
corr_df = train[NUM_COLS + ([TARGET] if pd.api.types.is_numeric_dtype(train[TARGET]) else [])]
corr = corr_df.corr(method="pearson")      # non-linear relation এর জন্য method="spearman"
plt.figure(figsize=(min(1 + .6 * len(corr), 18), min(1 + .5 * len(corr), 14)))
sns.heatmap(corr, annot=len(corr) <= 20, fmt=".2f", cmap="coolwarm", center=0, mask=np.triu(np.ones_like(corr, bool)))
plt.title("Correlation matrix"); plt.show()

if TARGET in corr:
    display(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).to_frame("corr_with_target"))

# Highly correlated feature pairs (multicollinearity)
pairs = corr.drop(index=TARGET, columns=TARGET, errors="ignore").abs().unstack()
pairs = pairs[pairs.index.get_level_values(0) < pairs.index.get_level_values(1)].sort_values(ascending=False)
print("Top correlated pairs:\n", pairs.head(10))

In [ ]:
# Pairplot (top-4 feature, sample) — বেশি column হলে slow
if TARGET in corr and len(NUM_COLS) >= 2:
    top4 = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).index[:4].tolist()
    sns.pairplot(train.sample(min(800, len(train)), random_state=SEED)[top4 + [TARGET]],
                 hue=TARGET if TASK == "classification" else None, corner=True, plot_kws={"s": 10, "alpha": .5})
    plt.show()

### 5.6 Train vs Test distribution + Adversarial Validation
Test data train এর মতো না হলে CV score আর leaderboard score মিলবে না।
**Adversarial validation:** একটা model কে train vs test আলাদা করতে বলি। AUC ≈ 0.5 → একই রকম ✅; AUC > 0.7 → distribution shift ⚠️

In [ ]:
if test is not None and NUM_COLS:
    fig, axes = grid(NUM_COLS)
    for ax, c in zip(axes, NUM_COLS):
        sns.kdeplot(train[c], ax=ax, label="train", fill=True, alpha=.3)
        sns.kdeplot(test[c], ax=ax, label="test", fill=True, alpha=.3)
        ax.legend(); ax.set_title(c)
    plt.tight_layout(); plt.show()

    adv = pd.concat([train[FEATURES].assign(_is_test=0), test[FEATURES].assign(_is_test=1)], ignore_index=True)
    for c in CAT_COLS:
        adv[c] = adv[c].astype("category").cat.codes
    adv_model = HistGradientBoostingClassifier(max_iter=200, random_state=SEED)
    adv_auc = cross_validate(adv_model, adv[FEATURES], adv["_is_test"], cv=5, scoring="roc_auc")["test_score"].mean()
    print(f"Adversarial validation AUC = {adv_auc:.4f}  {'✅ similar' if adv_auc < .6 else '⚠️ shift আছে — কোন feature দায়ী দেখো'}")
    adv_model.fit(adv[FEATURES], adv["_is_test"])
    pi = permutation_importance(adv_model, adv[FEATURES].iloc[::5], adv["_is_test"].iloc[::5], n_repeats=3, random_state=SEED)
    print("Shift এর জন্য দায়ী feature:", pd.Series(pi.importances_mean, FEATURES).sort_values(ascending=False).head(5).round(4).to_dict())

## 6. Outliers

In [ ]:
def outlier_report(df, cols):
    rows = []
    for c in cols:
        s = df[c].dropna()
        q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
        iqr_out = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
        z_out = (np.abs((s - s.mean()) / (s.std() + 1e-9)) > 3).sum()
        rows.append({"col": c, "iqr_outliers": iqr_out, "iqr_pct": iqr_out / len(s) * 100, "z>3": z_out,
                     "min": s.min(), "p1": s.quantile(.01), "p99": s.quantile(.99), "max": s.max()})
    return pd.DataFrame(rows).set_index("col").sort_values("iqr_pct", ascending=False)

display(outlier_report(train, NUM_COLS))

In [ ]:
# Option A: Capping / winsorizing (train এর percentile দিয়ে, test এও একই limit) — সবচেয়ে safe
CAP_LIMITS = {}
if CAP_OUTLIERS:
    for c in NUM_COLS:
        if train[c].nunique() > 20:          # binary/ordinal column clip করবো না
            lo, hi = train[c].quantile([.01, .99])
            CAP_LIMITS[c] = (lo, hi)
            train[c] = train[c].clip(lo, hi)
            if test is not None:
                test[c] = test[c].clip(lo, hi)
    print("Capped:", list(CAP_LIMITS))

# Option B: IsolationForest দিয়ে multivariate outlier row খোঁজা (শুধু দেখার জন্য; drop করলে সাবধানে)
iso = IsolationForest(contamination=0.01, random_state=SEED)
iso_flag = iso.fit_predict(train[NUM_COLS].fillna(train[NUM_COLS].median())) == -1
print(f"IsolationForest outlier rows: {iso_flag.sum()}")
# train = train[~iso_flag].reset_index(drop=True)   # ← drop করতে চাইলে uncomment (test row কখনো drop করবে না!)

## 7. Feature Engineering
**সবচেয়ে বেশি score বাড়ে এখান থেকেই।** Train + test একসাথে transform করি যাতে একই column তৈরি হয়।
- Date → year, month, dayofweek, is_weekend, days_since ...
- Skewed numeric → `log1p`
- Domain ratio / interaction → spend_per_product, income_per_age ...
- Frequency encoding, group aggregation (category অনুযায়ী mean, diff)
- Binning

In [ ]:
full = pd.concat([train.assign(_is_train=1), (test.assign(_is_train=0) if test is not None else None)], ignore_index=True)

# ---- 7.1 Date columns ----
DATE_COLS = [c for c in full.columns if is_text_col(full[c]) and re.search(r"date|time|_dt$|day", c, re.I)]
for c in DATE_COLS:
    d = pd.to_datetime(full[c], errors="coerce")
    full[f"{c}_year"] = d.dt.year
    full[f"{c}_month"] = d.dt.month
    full[f"{c}_day"] = d.dt.day
    full[f"{c}_dow"] = d.dt.dayofweek
    full[f"{c}_is_weekend"] = (d.dt.dayofweek >= 5).astype(int)
    full[f"{c}_quarter"] = d.dt.quarter
    full[f"{c}_days_since"] = (d.max() - d).dt.days
    # cyclical encoding (month 12 আর 1 কাছাকাছি বোঝাতে)
    full[f"{c}_month_sin"] = np.sin(2 * np.pi * d.dt.month / 12)
    full[f"{c}_month_cos"] = np.cos(2 * np.pi * d.dt.month / 12)
    full = full.drop(columns=c)
print("Date columns processed:", DATE_COLS)

# ---- 7.2 Ordinal mapping (order আছে এমন category) — নিজের data অনুযায়ী dict বানাও ----
ORDINAL_MAPS = {"plan": {"basic": 0, "standard": 1, "premium": 2}}
for c, m in ORDINAL_MAPS.items():
    if c in full:
        full[c] = full[c].map(m)

# ---- 7.3 Domain features (example — নিজের data অনুযায়ী লেখো) ----
def add_domain_features(df):
    if {"monthly_spend", "num_products"} <= set(df):
        df["spend_per_product"] = df["monthly_spend"] / (df["num_products"] + 1)
    if {"income", "age"} <= set(df):
        df["income_per_age"] = df["income"] / (df["age"] + 1)
    if {"tenure_months", "support_calls"} <= set(df):
        df["calls_per_year"] = df["support_calls"] / (df["tenure_months"] / 12 + 1)
    if {"monthly_spend", "tenure_months"} <= set(df):
        df["total_spend_est"] = df["monthly_spend"] * df["tenure_months"]
    return df

full = add_domain_features(full)

# ---- 7.4 Log transform skewed positive columns ----
_, num_now, cat_now = col_groups(full.drop(columns="_is_train"))
num_now = [c for c in num_now if not c.endswith("_is_missing")]
skewed = [c for c in num_now if full[c].nunique() > 20 and full[c].min() >= 0 and abs(full[c].skew()) > 1]
for c in skewed:
    full[f"log_{c}"] = np.log1p(full[c])
print("log1p added for:", skewed)

# ---- 7.5 Frequency encoding ----
for c in cat_now:
    full[f"{c}_freq"] = full[c].map(full[c].value_counts(normalize=True))

# ---- 7.6 Group aggregation: category অনুযায়ী numeric এর mean ও difference ----
GROUP_CATS = cat_now[:2]
GROUP_NUMS = [c for c in num_now if full[c].nunique() > 20][:3]
for g in GROUP_CATS:
    for n_ in GROUP_NUMS:
        grp_mean = full.groupby(g)[n_].transform("mean")
        full[f"{n_}_mean_by_{g}"] = grp_mean
        full[f"{n_}_diff_{g}_mean"] = full[n_] - grp_mean

# ---- 7.7 Binning example ----
if "age" in full:
    full["age_bin"] = pd.cut(full["age"], bins=[0, 25, 35, 50, 65, 200], labels=False)

# ---- split back ----
train_fe = full[full._is_train == 1].drop(columns="_is_train").reset_index(drop=True)
train_fe[TARGET] = train_fe[TARGET].astype(train[TARGET].dtype)     # concat এ NaN এর জন্য int → float হয়ে গিয়েছিল
test_fe = full[full._is_train == 0].drop(columns=["_is_train", TARGET], errors="ignore").reset_index(drop=True) if test is not None else None
FEATURES, NUM_COLS, CAT_COLS = col_groups(train_fe)
print(f"After FE → {len(FEATURES)} features ({len(NUM_COLS)} numeric, {len(CAT_COLS)} categorical)")

## 8. Encoding & Scaling
| Situation | Encoding |
|---|---|
| Low cardinality (≤ 15) nominal | **One-Hot** |
| Ordered category (low/med/high) | **Ordinal map** (Section 7.2) |
| High cardinality (city, zip, product_id) | **Target encoding** (CV-safe) / Frequency / Ordinal (tree এর জন্য) |
| Tree models | Ordinal encoding যথেষ্ট, scaling লাগে না |

| Scaler | কখন |
|---|---|
| StandardScaler | default (linear, SVM, KNN, NN, PCA) |
| MinMaxScaler | NN / image-like 0-1 range |
| RobustScaler | outlier অনেক |
| PowerTransformer / QuantileTransformer | heavily skewed → normal shape |

In [ ]:
# Scaler comparison on a skewed column
demo_col = max(NUM_COLS, key=lambda c: abs(train_fe[c].skew()) if train_fe[c].nunique() > 20 else 0)
vals = train_fe[[demo_col]].dropna()
scalers = {"original": None, "Standard": StandardScaler(), "MinMax": MinMaxScaler(), "Robust": RobustScaler(),
           "Power(Yeo-Johnson)": PowerTransformer(), "Quantile(normal)": QuantileTransformer(output_distribution="normal", random_state=SEED)}
fig, axes = plt.subplots(1, len(scalers), figsize=(22, 3))
for ax, (name, sc) in zip(axes, scalers.items()):
    v = vals.values if sc is None else sc.fit_transform(vals)
    sns.histplot(v.ravel(), ax=ax, bins=40, kde=True); ax.set_title(name)
fig.suptitle(f"Scaling comparison on '{demo_col}'", y=1.05); plt.show()

In [ ]:
# Manual encoding toolbox (EDA/quick test এর জন্য; আসল কাজ নিচের Pipeline এ)
if CAT_COLS:
    c0 = CAT_COLS[0]
    print("One-hot (pd.get_dummies):", pd.get_dummies(train_fe[[c0]], dummy_na=True).shape)
    print("Label encoding:", LabelEncoder().fit_transform(train_fe[c0].astype(str))[:10])

In [ ]:
LOW_CARD = [c for c in CAT_COLS if train_fe[c].nunique() <= HIGH_CARD_THRESH]
HIGH_CARD = [c for c in CAT_COLS if c not in LOW_CARD]
print("Low-card (one-hot):", LOW_CARD, "| High-card (target enc):", HIGH_CARD)

def make_preprocessor(kind="linear", num_cols=None, low_card=None, high_card=None):
    """kind='linear' → impute+scale+onehot+target-enc ; kind='tree' → impute+ordinal (no scaling)."""
    num_cols = NUM_COLS if num_cols is None else num_cols
    low_card = LOW_CARD if low_card is None else low_card
    high_card = HIGH_CARD if high_card is None else high_card
    target_type = "continuous" if TASK == "regression" else ("binary" if IS_BINARY else "multiclass")
    if kind == "linear":
        transformers = [
            ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                              ("sc", StandardScaler())]), num_cols),
            ("ohe", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                              ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=False))]), low_card),
            ("te", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                             ("te", TargetEncoder(target_type=target_type, random_state=SEED)),
                             ("sc", StandardScaler())]), high_card),
        ]
    else:
        transformers = [
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                              ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]),
             low_card + high_card),
        ]
    transformers = [t for t in transformers if len(t[2])]
    return ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)

X = train_fe[FEATURES].copy()
X_test = test_fe[FEATURES].copy() if test_fe is not None else None

y_raw = train_fe[TARGET]
LABEL_ENC = None
if TASK == "classification":
    LABEL_ENC = LabelEncoder().fit(y_raw)
    y = pd.Series(LABEL_ENC.transform(y_raw), name=TARGET)
    print("Classes:", dict(enumerate(LABEL_ENC.classes_)))
else:
    y = y_raw.astype(float)

print("Linear-prep output shape:", make_preprocessor("linear").fit_transform(X, y).shape)
print("Tree-prep output shape:  ", make_preprocessor("tree").fit_transform(X, y).shape)

## 9. Feature Selection
কয়েকটা method একসাথে চালিয়ে দেখি কোন feature গুলো বারবার important আসে।
(Tree-prep এ প্রতিটা original feature = ১টা column, তাই নাম সরাসরি মেলে।)

In [ ]:
tree_prep = make_preprocessor("tree").fit(X, y)
Xt = pd.DataFrame(tree_prep.transform(X), columns=tree_prep.get_feature_names_out())
fs = pd.DataFrame(index=Xt.columns)

# 9.1 Variance threshold
vt = VarianceThreshold(threshold=1e-4).fit(Xt)
fs["low_variance"] = ~vt.get_support()

# 9.2 Mutual information (non-linear relation ধরতে পারে)
mi_fn = mutual_info_classif if TASK == "classification" else mutual_info_regression
fs["mutual_info"] = mi_fn(Xt, y, random_state=SEED)

# 9.3 ANOVA F-test (linear relation)
f_fn = f_classif if TASK == "classification" else f_regression
fs["f_score"] = np.nan_to_num(f_fn(Xt, y)[0])

# 9.4 Tree importance (RandomForest)
rf_fs = (RandomForestClassifier if TASK == "classification" else RandomForestRegressor)(n_estimators=200, n_jobs=-1, random_state=SEED)
fs["rf_importance"] = rf_fs.fit(Xt, y).feature_importances_

# 9.5 L1 regularization (Lasso → unimportant feature এর coef = 0)
Xs = StandardScaler().fit_transform(Xt)
l1 = (LogisticRegression(penalty="l1", solver="liblinear", C=0.1, random_state=SEED) if TASK == "classification"
      else Lasso(alpha=0.01 * y.std(), random_state=SEED))
l1.fit(Xs, y)
fs["l1_coef_abs"] = np.abs(l1.coef_).mean(axis=0) if l1.coef_.ndim > 1 else np.abs(l1.coef_)

# Rank-average → সবগুলো method এর consensus
rank_cols = ["mutual_info", "f_score", "rf_importance", "l1_coef_abs"]
fs["avg_rank"] = fs[rank_cols].rank(ascending=False).mean(axis=1)
fs = fs.sort_values("avg_rank")
display(fs.style.background_gradient(cmap="Greens", subset=rank_cols))

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, .25 * len(fs))))
fs["mutual_info"].sort_values().plot.barh(ax=axes[0], title="Mutual Information")
fs["rf_importance"].sort_values().plot.barh(ax=axes[1], title="RandomForest importance", color="orange")
plt.tight_layout(); plt.show()

In [ ]:
# 9.6 Highly correlated feature drop (|r| > 0.95 হলে একটা রাখো)
cm = Xt.corr().abs()
upper = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
corr_drop = [c for c in upper.columns if (upper[c] > 0.95).any()]
print("Highly correlated (drop candidates):", corr_drop)

# 9.7 RFECV — recursive feature elimination with CV (slow হলে step বাড়াও)
rfe_est = (LogisticRegression(max_iter=2000) if TASK == "classification" else Ridge())
rfecv = RFECV(rfe_est, step=max(1, Xs.shape[1] // 20), cv=3, scoring=METRIC, n_jobs=-1).fit(Xs, y)
print(f"RFECV optimal #features = {rfecv.n_features_}")
plt.plot(rfecv.cv_results_["n_features"], rfecv.cv_results_["mean_test_score"], marker="o")
plt.xlabel("# features"); plt.ylabel(METRIC); plt.title("RFECV"); plt.show()

In [ ]:
# ---- Final decision ----
USE_FEATURE_SELECTION = False     # True করলে নিচের list ব্যবহার হবে
TOP_K = max(10, int(len(fs) * 0.7))
SELECTED = [c for c in fs.index[:TOP_K] if c not in corr_drop and not fs.loc[c, "low_variance"]]
if USE_FEATURE_SELECTION:
    NUM_COLS = [c for c in NUM_COLS if c in SELECTED]
    LOW_CARD = [c for c in LOW_CARD if c in SELECTED]
    HIGH_CARD = [c for c in HIGH_CARD if c in SELECTED]
    FEATURES = NUM_COLS + LOW_CARD + HIGH_CARD
    X = X[FEATURES]; X_test = X_test[FEATURES] if X_test is not None else None
print(f"Using {len(FEATURES)} features")

## 10. Dimensionality Reduction (visualization + optional feature)

In [ ]:
lin_prep = make_preprocessor("linear").fit(X, y)
X_lin = lin_prep.transform(X)
pca = PCA(random_state=SEED).fit(X_lin)
cum = np.cumsum(pca.explained_variance_ratio_)
n95 = int(np.searchsorted(cum, .95) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(range(1, len(cum) + 1), cum, marker="."); axes[0].axhline(.95, ls="--", c="r")
axes[0].set_title(f"PCA cumulative variance (95% → {n95} comps)")
p2 = PCA(2, random_state=SEED).fit_transform(X_lin)
axes[1].scatter(p2[:, 0], p2[:, 1], c=y, cmap="coolwarm", s=5, alpha=.6); axes[1].set_title("PCA 2D")
idx = np.random.RandomState(SEED).choice(len(X_lin), min(1500, len(X_lin)), replace=False)
ts = TSNE(2, random_state=SEED, perplexity=30, init="pca").fit_transform(X_lin[idx])
axes[2].scatter(ts[:, 0], ts[:, 1], c=y.iloc[idx], cmap="coolwarm", s=5, alpha=.6); axes[2].set_title("t-SNE 2D (sample)")
plt.tight_layout(); plt.show()

## 11. Model Zoo — সব model একসাথে CV দিয়ে compare
**CV strategy বাছাই:**
- Classification → `StratifiedKFold` (class ratio ঠিক রাখে)
- Regression → `KFold`
- একই user/patient এর একাধিক row → `GroupKFold` (না হলে leakage!)
- Time series → `TimeSeriesSplit` (ভবিষ্যৎ দিয়ে অতীত predict করা যাবে না)

In [ ]:
CV = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED) if TASK == "classification" else KFold(N_FOLDS, shuffle=True, random_state=SEED)
# CV = GroupKFold(N_FOLDS)            # cross_validate(..., groups=train_fe["user_id"])
# CV = TimeSeriesSplit(N_FOLDS)

def get_models():
    """(name → (estimator, preprocessing kind)). Optional library না থাকলে auto skip।"""
    big = len(X) > SLOW_MODEL_MAX_ROWS
    if TASK == "classification":
        m = {
            "Dummy": (DummyClassifier(strategy="prior"), "tree"),
            "LogisticRegression": (LogisticRegression(max_iter=2000, C=1.0), "linear"),
            "LDA": (LinearDiscriminantAnalysis(), "linear"),
            "GaussianNB": (GaussianNB(), "linear"),
            "KNN": (KNeighborsClassifier(n_neighbors=15), "linear"),
            "SVM_RBF": (SVC(probability=True, C=1.0, random_state=SEED), "linear"),
            "DecisionTree": (DecisionTreeClassifier(max_depth=6, random_state=SEED), "tree"),
            "RandomForest": (RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED), "tree"),
            "ExtraTrees": (ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=SEED), "tree"),
            "AdaBoost": (AdaBoostClassifier(n_estimators=200, random_state=SEED), "tree"),
            "GradientBoosting": (GradientBoostingClassifier(random_state=SEED), "tree"),
            "HistGradientBoosting": (HistGradientBoostingClassifier(random_state=SEED), "tree"),
            "MLP": (MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=500, random_state=SEED), "linear"),
        }
        if xgb: m["XGBoost"] = (xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5, subsample=.8, colsample_bytree=.8, random_state=SEED, n_jobs=-1, verbosity=0), "tree")
        if lgb: m["LightGBM"] = (lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=.8, subsample_freq=1, colsample_bytree=.8, random_state=SEED, verbose=-1), "tree")
        if cb: m["CatBoost"] = (cb.CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0), "tree")
    else:
        m = {
            "Dummy": (DummyRegressor(), "tree"),
            "LinearRegression": (LinearRegression(), "linear"),
            "Ridge": (Ridge(alpha=1.0), "linear"),
            "Lasso": (Lasso(alpha=0.001 * y.std(), max_iter=5000), "linear"),
            "ElasticNet": (ElasticNet(alpha=0.001 * y.std(), l1_ratio=.5, max_iter=5000), "linear"),
            "KNN": (KNeighborsRegressor(n_neighbors=15), "linear"),
            "SVR": (SVR(C=1.0), "linear"),
            "DecisionTree": (DecisionTreeRegressor(max_depth=8, random_state=SEED), "tree"),
            "RandomForest": (RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED), "tree"),
            "ExtraTrees": (ExtraTreesRegressor(n_estimators=300, n_jobs=-1, random_state=SEED), "tree"),
            "AdaBoost": (AdaBoostRegressor(n_estimators=200, random_state=SEED), "tree"),
            "GradientBoosting": (GradientBoostingRegressor(random_state=SEED), "tree"),
            "HistGradientBoosting": (HistGradientBoostingRegressor(random_state=SEED), "tree"),
            "MLP": (MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=1000, random_state=SEED), "linear"),
        }
        if xgb: m["XGBoost"] = (xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, subsample=.8, colsample_bytree=.8, random_state=SEED, n_jobs=-1, verbosity=0), "tree")
        if lgb: m["LightGBM"] = (lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=.8, subsample_freq=1, colsample_bytree=.8, random_state=SEED, verbose=-1), "tree")
        if cb: m["CatBoost"] = (cb.CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0), "tree")
    if big:
        for k in ["KNN", "SVM_RBF", "SVR", "GradientBoosting"]:
            m.pop(k, None)
    return m

def make_pipe(est, kind):
    return Pipeline([("prep", make_preprocessor(kind)), ("model", est)])

results = []
for name, (est, kind) in get_models().items():
    t0 = time.time()
    cvres = cross_validate(make_pipe(est, kind), X, y, cv=CV, scoring=METRIC, n_jobs=1, return_train_score=True)
    results.append({"model": name, "cv_mean": cvres["test_score"].mean(), "cv_std": cvres["test_score"].std(),
                    "train_mean": cvres["train_score"].mean(), "time_s": time.time() - t0})
    print(f"{name:22s} {METRIC}: {results[-1]['cv_mean']:.4f} ± {results[-1]['cv_std']:.4f}  ({results[-1]['time_s']:.1f}s)")

leaderboard = pd.DataFrame(results).sort_values("cv_mean", ascending=False).reset_index(drop=True)
leaderboard["overfit_gap"] = leaderboard["train_mean"] - leaderboard["cv_mean"]
display(leaderboard)

In [ ]:
plt.figure(figsize=(10, .4 * len(leaderboard) + 1))
plt.barh(leaderboard["model"][::-1], leaderboard["cv_mean"][::-1], xerr=leaderboard["cv_std"][::-1], color="teal", alpha=.8)
lo = leaderboard["cv_mean"].min()
plt.xlim(lo - abs(lo) * 0.05 if lo > 0 else lo * 1.05, None)
plt.title(f"Model comparison — {N_FOLDS}-fold CV {METRIC}"); plt.show()

TOP_MODELS = leaderboard["model"].head(3).tolist()
print("Top 3:", TOP_MODELS)

## 12. Class Imbalance Handling (classification)
- `class_weight="balanced"` (LogReg, RF, SVM, LightGBM) / `scale_pos_weight` (XGBoost) / `auto_class_weights="Balanced"` (CatBoost)
- **SMOTE** oversampling — শুধু train fold এ! তাই `imblearn.pipeline.Pipeline` ব্যবহার করতে হয়
- Metric: accuracy না, **F1 / ROC-AUC / PR-AUC / balanced accuracy** দেখো
- Threshold tuning (Section 15)

In [ ]:
if TASK == "classification":
    vc = y.value_counts(normalize=True)
    print("Minority class share:", round(vc.min(), 3))
    comp = {}
    comp["RF (no weight)"] = make_pipe(RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED), "tree")
    comp["RF class_weight=balanced"] = make_pipe(RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=SEED), "tree")
    if imblearn is not None:
        from imblearn.pipeline import Pipeline as ImbPipeline
        from imblearn.over_sampling import SMOTE
        comp["RF + SMOTE"] = ImbPipeline([("prep", make_preprocessor("tree")), ("smote", SMOTE(random_state=SEED)),
                                          ("model", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED))])
    for k, p in comp.items():
        s = cross_validate(p, X, y, cv=CV, scoring=["f1_macro", "balanced_accuracy", METRIC] if METRIC not in ["f1_macro", "balanced_accuracy"] else ["f1_macro", "balanced_accuracy"])
        print(f"{k:28s} " + " | ".join(f"{m[5:]}={v.mean():.4f}" for m, v in s.items() if m.startswith("test_")))

## 13. Hyperparameter Tuning
### 13.1 RandomizedSearchCV (যেকোনো sklearn model)

In [ ]:
rf_pipe = make_pipe((RandomForestClassifier if TASK == "classification" else RandomForestRegressor)(n_jobs=-1, random_state=SEED), "tree")
param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 6, 10, 16],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", 0.5, 0.8],
}
rs = RandomizedSearchCV(rf_pipe, param_dist, n_iter=10, cv=CV, scoring=METRIC, random_state=SEED, n_jobs=1, verbose=0)
rs.fit(X, y)
print(f"Best RF {METRIC}: {rs.best_score_:.4f}\nBest params: {rs.best_params_}")

### 13.2 Optuna (Bayesian search — boosting model এর জন্য best)

In [ ]:
BEST_GBM_PARAMS = {}
if optuna is not None:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        if lgb is not None:
            params = dict(
                n_estimators=trial.suggest_int("n_estimators", 200, 1500, step=100),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                num_leaves=trial.suggest_int("num_leaves", 8, 128, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 12),
                min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
                subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
                colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
                reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
                reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
                random_state=SEED, verbose=-1)
            est = (lgb.LGBMClassifier if TASK == "classification" else lgb.LGBMRegressor)(**params)
        else:
            params = dict(
                max_iter=trial.suggest_int("max_iter", 100, 800, step=100),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 8, 128, log=True),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 100),
                l2_regularization=trial.suggest_float("l2_regularization", 1e-8, 10, log=True),
                random_state=SEED)
            est = (HistGradientBoostingClassifier if TASK == "classification" else HistGradientBoostingRegressor)(**params)
        return cross_validate(make_pipe(est, "tree"), X, y, cv=CV, scoring=METRIC)["test_score"].mean()

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, timeout=600, show_progress_bar=False)
    BEST_GBM_PARAMS = study.best_params
    print(f"Best Optuna {METRIC}: {study.best_value:.4f}\n{BEST_GBM_PARAMS}")

    try:
        optuna.visualization.matplotlib.plot_optimization_history(study); plt.show()
        optuna.visualization.matplotlib.plot_param_importances(study); plt.show()
    except Exception as e:
        print("Optuna plot skipped:", e)

## 14. K-Fold Training → OOF (out-of-fold) + Test predictions
প্রতিটা fold এ train → validation predict (OOF) → test predict। Test prediction = সব fold এর average।
OOF score = সবচেয়ে বিশ্বাসযোগ্য CV score, আর OOF দিয়েই ensemble weight / threshold ঠিক করা হয়।

In [ ]:
def score_oof(y_true, pred, metric=METRIC):
    """pred: classification → probability matrix (n, n_classes); regression → 1D. Higher = better (sklearn convention)."""
    if TASK == "classification":
        labels = pred.argmax(1)
        if metric == "accuracy": return metrics.accuracy_score(y_true, labels)
        if metric == "balanced_accuracy": return metrics.balanced_accuracy_score(y_true, labels)
        if metric.startswith("f1"):
            avg = metric.split("_")[1] if "_" in metric else "binary"
            return metrics.f1_score(y_true, labels, average=avg)
        if metric == "roc_auc": return metrics.roc_auc_score(y_true, pred[:, 1])
        if metric.startswith("roc_auc_ov"): return metrics.roc_auc_score(y_true, pred, multi_class=metric[-3:])
        if metric == "neg_log_loss": return -metrics.log_loss(y_true, pred)
        if metric == "average_precision": return metrics.average_precision_score(y_true, pred[:, 1])
    else:
        if metric == "neg_root_mean_squared_error": return -metrics.root_mean_squared_error(y_true, pred)
        if metric == "neg_mean_absolute_error": return -metrics.mean_absolute_error(y_true, pred)
        if metric == "neg_mean_squared_error": return -metrics.mean_squared_error(y_true, pred)
        if metric == "r2": return metrics.r2_score(y_true, pred)
    raise ValueError(f"score_oof এ '{metric}' add করো")


def run_kfold(pipe, X, y, X_test=None, cv=CV, name="model"):
    n_out = N_CLASSES if TASK == "classification" else 1
    oof = np.zeros((len(X), n_out))
    test_pred = np.zeros((len(X_test), n_out)) if X_test is not None else None
    for fold, (tr, va) in enumerate(cv.split(X, y)):
        m = clone(pipe).fit(X.iloc[tr], y.iloc[tr])
        if TASK == "classification":
            oof[va] = m.predict_proba(X.iloc[va])
            if X_test is not None: test_pred += m.predict_proba(X_test) / cv.get_n_splits()
        else:
            oof[va, 0] = m.predict(X.iloc[va])
            if X_test is not None: test_pred[:, 0] += m.predict(X_test) / cv.get_n_splits()
    if TASK == "regression":
        oof = oof[:, 0]; test_pred = test_pred[:, 0] if test_pred is not None else None
    print(f"{name:22s} OOF {METRIC}: {score_oof(y, oof):.4f}")
    return oof, test_pred


# Top-3 model + tuned model গুলো
all_models = get_models()
final_pipes = {n: make_pipe(*all_models[n]) for n in TOP_MODELS if n != "Dummy"}
final_pipes["RF_tuned"] = rs.best_estimator_
if BEST_GBM_PARAMS:
    if lgb is not None:
        est = (lgb.LGBMClassifier if TASK == "classification" else lgb.LGBMRegressor)(**BEST_GBM_PARAMS, subsample_freq=1, random_state=SEED, verbose=-1)
    else:
        est = (HistGradientBoostingClassifier if TASK == "classification" else HistGradientBoostingRegressor)(**BEST_GBM_PARAMS, random_state=SEED)
    final_pipes["GBM_optuna"] = make_pipe(est, "tree")

OOF, TEST_PRED = {}, {}
for name, pipe in final_pipes.items():
    OOF[name], TEST_PRED[name] = run_kfold(pipe, X, y, X_test, name=name)

## 15. Ensembling
- **Simple average** — প্রায় সবসময় single model থেকে ভালো
- **Weighted average** — OOF দিয়ে weight optimize
- **Stacking** — OOF prediction গুলোকে feature বানিয়ে একটা meta-model (LogReg/Ridge)
- Ensemble এ **different type** model (tree + linear + NN) রাখলে বেশি লাভ

In [ ]:
from scipy.optimize import minimize

names = list(OOF)
ens_scores = {n: score_oof(y, OOF[n]) for n in names}

# Simple average
oof_avg = np.mean([OOF[n] for n in names], axis=0)
test_avg = np.mean([TEST_PRED[n] for n in names], axis=0) if X_test is not None else None
ens_scores["AVG"] = score_oof(y, oof_avg)

# Weighted (scipy minimize on -score; Nelder-Mead, weights normalized)
def _neg(w):
    w = np.abs(w) / np.abs(w).sum()
    return -score_oof(y, sum(wi * OOF[n] for wi, n in zip(w, names)))
res = minimize(_neg, np.ones(len(names)) / len(names), method="Nelder-Mead", options={"maxiter": 300})
W = np.abs(res.x) / np.abs(res.x).sum()
oof_w = sum(wi * OOF[n] for wi, n in zip(W, names))
test_w = sum(wi * TEST_PRED[n] for wi, n in zip(W, names)) if X_test is not None else None
ens_scores["WEIGHTED"] = score_oof(y, oof_w)
print("Weights:", dict(zip(names, W.round(3))))

# Stacking (meta-model on OOF; meta এর নিজের score ও CV দিয়ে)
def stack_feats(d):
    if TASK == "classification":
        return np.hstack([d[n][:, 1:] if IS_BINARY else d[n] for n in names])
    return np.column_stack([d[n] for n in names])
meta = LogisticRegression(max_iter=2000, C=1.0) if TASK == "classification" else Ridge(alpha=1.0)
S_tr = stack_feats(OOF)
oof_stack = cross_val_predict(meta, S_tr, y, cv=CV, method="predict_proba" if TASK == "classification" else "predict")
ens_scores["STACK"] = score_oof(y, oof_stack)
test_stack = meta.fit(S_tr, y).predict_proba(stack_feats(TEST_PRED)) if (X_test is not None and TASK == "classification") else \
             (meta.fit(S_tr, y).predict(stack_feats(TEST_PRED)) if X_test is not None else None)

ens_table = pd.Series(ens_scores).sort_values(ascending=False).to_frame(f"OOF {METRIC}")
display(ens_table)

CANDIDATES = {**{n: (OOF[n], TEST_PRED[n]) for n in names}, "AVG": (oof_avg, test_avg), "WEIGHTED": (oof_w, test_w), "STACK": (oof_stack, test_stack)}
BEST_NAME = ens_table.index[0]
BEST_OOF, BEST_TEST = CANDIDATES[BEST_NAME]
print("🏆 Best:", BEST_NAME)

In [ ]:
# Threshold tuning (binary + F1/accuracy type metric হলে) — default 0.5 সবসময় best না
BEST_THRESHOLD = 0.5
if TASK == "classification" and IS_BINARY:
    ths = np.linspace(0.05, 0.95, 91)
    f1s = [metrics.f1_score(y, (BEST_OOF[:, 1] >= t).astype(int)) for t in ths]
    BEST_THRESHOLD = ths[int(np.argmax(f1s))]
    plt.plot(ths, f1s); plt.axvline(BEST_THRESHOLD, c="r", ls="--"); plt.title(f"F1 vs threshold (best={BEST_THRESHOLD:.2f})"); plt.show()
    print(f"F1 @0.5 = {metrics.f1_score(y, (BEST_OOF[:, 1] >= .5).astype(int)):.4f} | F1 @{BEST_THRESHOLD:.2f} = {max(f1s):.4f}")

## 16. Evaluation Plots (OOF predictions এর উপর)

In [ ]:
if TASK == "classification":
    pred_lbl = (BEST_OOF[:, 1] >= BEST_THRESHOLD).astype(int) if IS_BINARY else BEST_OOF.argmax(1)
    class_names = [str(c) for c in LABEL_ENC.classes_]
    print(metrics.classification_report(y, pred_lbl, target_names=class_names, digits=4))

    fig, axes = plt.subplots(1, 3 if IS_BINARY else 2, figsize=(18 if IS_BINARY else 12, 4.5))
    metrics.ConfusionMatrixDisplay.from_predictions(y, pred_lbl, display_labels=class_names, ax=axes[0], cmap="Blues", colorbar=False)
    metrics.ConfusionMatrixDisplay.from_predictions(y, pred_lbl, display_labels=class_names, ax=axes[1], cmap="Greens", normalize="true", colorbar=False)
    axes[0].set_title("Confusion matrix"); axes[1].set_title("Normalized (recall per class)")
    if IS_BINARY:
        for n, (o, _) in CANDIDATES.items():
            fpr, tpr, _ = metrics.roc_curve(y, o[:, 1])
            axes[2].plot(fpr, tpr, label=f"{n} ({metrics.auc(fpr, tpr):.3f})", lw=2 if n == BEST_NAME else 1)
        axes[2].plot([0, 1], [0, 1], "k--"); axes[2].legend(fontsize=8); axes[2].set_title("ROC curves")
    plt.tight_layout(); plt.show()

    if IS_BINARY:
        fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
        metrics.PrecisionRecallDisplay.from_predictions(y, BEST_OOF[:, 1], ax=axes[0]); axes[0].set_title("Precision-Recall")
        from sklearn.calibration import CalibrationDisplay
        CalibrationDisplay.from_predictions(y, BEST_OOF[:, 1], n_bins=10, ax=axes[1]); axes[1].set_title("Calibration")
        sns.histplot(x=BEST_OOF[:, 1], hue=y, bins=50, ax=axes[2], element="step", stat="density", common_norm=False)
        axes[2].set_title("Predicted prob by true class")
        plt.tight_layout(); plt.show()
else:
    res_ = y - BEST_OOF
    print(f"RMSE={metrics.root_mean_squared_error(y, BEST_OOF):.4f} | MAE={metrics.mean_absolute_error(y, BEST_OOF):.4f} | R2={metrics.r2_score(y, BEST_OOF):.4f}")
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    axes[0].scatter(y, BEST_OOF, s=5, alpha=.4); axes[0].plot([y.min(), y.max()], [y.min(), y.max()], "r--"); axes[0].set(xlabel="actual", ylabel="predicted", title="Actual vs Predicted")
    sns.histplot(res_, kde=True, ax=axes[1]); axes[1].set_title("Residual distribution")
    axes[2].scatter(BEST_OOF, res_, s=5, alpha=.4); axes[2].axhline(0, c="r"); axes[2].set(xlabel="predicted", ylabel="residual", title="Residual vs Predicted")
    plt.tight_layout(); plt.show()

In [ ]:
# Learning curve — more data দিলে লাভ হবে? (train-val gap বড় = overfit; দুটোই low = underfit)
best_single = max(names, key=lambda n: ens_scores[n])
sizes, tr_sc, va_sc = learning_curve(final_pipes[best_single], X, y, cv=CV, scoring=METRIC,
                                     train_sizes=np.linspace(.1, 1, 6), n_jobs=1)
plt.plot(sizes, tr_sc.mean(1), "o-", label="train"); plt.plot(sizes, va_sc.mean(1), "o-", label="validation")
plt.fill_between(sizes, va_sc.mean(1) - va_sc.std(1), va_sc.mean(1) + va_sc.std(1), alpha=.2)
plt.legend(); plt.title(f"Learning curve — {best_single}"); plt.xlabel("train size"); plt.ylabel(METRIC); plt.show()

## 17. Model Interpretation
- **Built-in importance** (tree): দ্রুত, কিন্তু high-cardinality feature কে বেশি গুরুত্ব দেয়
- **Permutation importance**: feature shuffle করলে score কত কমে — বেশি reliable
- **SHAP**: প্রতিটা prediction এ প্রতিটা feature এর contribution (direction সহ)
- **PDP**: একটা feature বাড়লে prediction কিভাবে বদলায়

In [ ]:
# Interpretation এর জন্য একটা tree model (LightGBM > XGB > RF priority)
interp_name = next((n for n in ["GBM_optuna", "LightGBM", "XGBoost", "CatBoost", "RandomForest", "RF_tuned"] if n in final_pipes), best_single)
X_tr_i, X_va_i, y_tr_i, y_va_i = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y if TASK == "classification" else None)
interp_pipe = clone(final_pipes[interp_name]).fit(X_tr_i, y_tr_i)
prep_i, model_i = interp_pipe.named_steps["prep"], interp_pipe.named_steps["model"]
feat_names_i = prep_i.get_feature_names_out()
print("Interpreting:", interp_name)

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, .28 * len(feat_names_i))))
if hasattr(model_i, "feature_importances_"):
    pd.Series(model_i.feature_importances_, feat_names_i).sort_values().tail(25).plot.barh(ax=axes[0], title=f"{interp_name} built-in importance")
pi = permutation_importance(interp_pipe, X_va_i, y_va_i, scoring=METRIC, n_repeats=5, random_state=SEED, n_jobs=1)
pi_s = pd.Series(pi.importances_mean, X.columns).sort_values()
pi_s.tail(25).plot.barh(ax=axes[1], xerr=pd.Series(pi.importances_std, X.columns)[pi_s.index].tail(25).values, color="orange", title="Permutation importance (validation)")
plt.tight_layout(); plt.show()

In [ ]:
if shap is not None:
    Xv_t = pd.DataFrame(prep_i.transform(X_va_i), columns=feat_names_i).iloc[:1000]
    try:
        explainer = shap.TreeExplainer(model_i)
        sv = explainer(Xv_t)
    except Exception as e:
        print("TreeExplainer failed, using generic Explainer:", e)
        f = (lambda d: model_i.predict_proba(d)[:, 1]) if (TASK == "classification" and IS_BINARY) else model_i.predict
        explainer = shap.Explainer(f, Xv_t.iloc[:100]); sv = explainer(Xv_t.iloc[:300])
    if sv.values.ndim == 3:                     # multiclass/binary 2-output → positive (last) class
        sv = sv[:, :, -1]
    shap.plots.beeswarm(sv, max_display=20, show=False); plt.title("SHAP summary"); plt.show()
    shap.plots.bar(sv, max_display=20, show=False); plt.show()
    top_feat = pd.Series(np.abs(sv.values).mean(0), feat_names_i).idxmax()
    shap.plots.scatter(sv[:, top_feat], color=sv, show=False); plt.show()
    shap.plots.waterfall(sv[0], max_display=12, show=False); plt.title("Why this single prediction?"); plt.show()

In [ ]:
# Partial Dependence (top-3 numeric feature by permutation importance)
top_num = [c for c in pi_s.index[::-1] if c in NUM_COLS][:3]
if top_num:
    fig, ax = plt.subplots(figsize=(15, 4))
    X_pdp = X_va_i.fillna(X_tr_i.median(numeric_only=True))      # PDP grid NaN সহ্য করে না
    X_pdp[NUM_COLS] = X_pdp[NUM_COLS].astype(float)              # integer column এ PDP error দেয়
    PartialDependenceDisplay.from_estimator(interp_pipe, X_pdp, top_num, ax=ax, kind="both", subsample=300,
                                            random_state=SEED, target=(1 if (TASK == "classification" and not IS_BINARY) else None))
    plt.tight_layout(); plt.show()

## 18. Submission

In [ ]:
if X_test is not None:
    if TASK == "classification":
        PROBA_METRICS = {"roc_auc", "neg_log_loss", "average_precision", "roc_auc_ovr", "roc_auc_ovo"}
        if METRIC in PROBA_METRICS:
            final_pred = BEST_TEST[:, 1] if IS_BINARY else BEST_TEST          # probability submit
        else:
            idx = (BEST_TEST[:, 1] >= BEST_THRESHOLD).astype(int) if IS_BINARY else BEST_TEST.argmax(1)
            final_pred = LABEL_ENC.inverse_transform(idx)                      # original label এ ফেরত
    else:
        final_pred = BEST_TEST

    if np.ndim(final_pred) == 2:                                   # multiclass probability → এক class এক column
        sub = pd.DataFrame(final_pred, columns=[str(c) for c in LABEL_ENC.classes_])
        if ID_COL: sub.insert(0, ID_COL, test[ID_COL].values)
        if sample_sub is not None: sub.columns = sample_sub.columns
    elif sample_sub is not None:
        sub = sample_sub.copy()
        if ID_COL and ID_COL in sub:
            sub = sub[[ID_COL]].merge(pd.DataFrame({ID_COL: test[ID_COL].values, "_p": list(final_pred)}), on=ID_COL, how="left")
            sub = sub.rename(columns={"_p": sample_sub.columns[1]})
        else:
            sub[sample_sub.columns[1]] = final_pred
    else:
        sub = pd.DataFrame({ID_COL or "id": test[ID_COL].values if ID_COL else np.arange(len(final_pred)), TARGET: final_pred})

    out_path = OUTPUT_DIR / "submission.csv"
    sub.to_csv(out_path, index=False)

    # ---- Sanity checks ----
    print("Saved →", out_path, sub.shape)
    if sample_sub is not None:
        assert sub.shape == sample_sub.shape, "shape sample_submission এর সাথে মিলছে না!"
        assert list(sub.columns) == list(sample_sub.columns), "column নাম মিলছে না!"
    assert sub.iloc[:, 1].notna().all(), "prediction এ NaN আছে!"
    display(sub.head())
    print(sub.iloc[:, 1].describe() if TASK == "regression" or METRIC in {"roc_auc", "neg_log_loss"} else sub.iloc[:, 1].value_counts(normalize=True))

In [ ]:
# Final model save (পরে reuse / inference এর জন্য)
import joblib
final_model = clone(final_pipes[best_single]).fit(X, y)
joblib.dump({"model": final_model, "features": FEATURES, "label_encoder": LABEL_ENC}, OUTPUT_DIR / "final_model.joblib")
print("Model saved →", OUTPUT_DIR / "final_model.joblib")
# loaded = joblib.load(OUTPUT_DIR / "final_model.joblib"); loaded["model"].predict(X_test)

## ☁️ Colab: Submission download + Drive এ backup

In [ ]:
SAVE_TO_DRIVE = False                                  # True → outputs folder Google Drive এ copy হবে
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/datathon/outputs/tabular"

if IN_COLAB:
    from google.colab import files, drive
    sub_path = OUTPUT_DIR / "submission.csv"
    if sub_path.exists():
        files.download(str(sub_path))                  # browser এ download হবে
    if SAVE_TO_DRIVE:
        drive.mount("/content/drive")
        shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
        print("Saved to Drive →", DRIVE_OUTPUT_DIR)
else:
    print("Local run — files আছে:", OUTPUT_DIR.resolve())

## ✅ Checklist — score বাড়ানোর জন্য পরের ধাপ
1. **CV ঠিক আছে তো?** CV score আর public LB মিলছে কিনা — না মিললে CV strategy / leakage চেক
2. **Feature engineering এ আরো সময় দাও** — domain ratio, group-agg, date features, interaction
3. Target **skewed** (regression) → `log1p(y)` এ train করে `expm1` দিয়ে ফেরত
4. **GBM tuning** (Optuna trials বাড়াও), `n_estimators` বড় + `learning_rate` ছোট
5. **Diverse ensemble** — LightGBM + CatBoost + XGBoost + LogReg/NN
6. **Multiple seeds** average (seed averaging)
7. Classification metric F1 হলে **threshold tune** করো
8. Submission format **sample_submission এর সাথে হুবহু মিলাও**